# NB21 — Profit-Margin Conclusion (bookend to NB19 + NB20)

**What this is.** The bookend to the profit-margin factor. Consumes `outputs/nb19_margin.json` (the *point* — where each hospital's operating margin sits) **and** `outputs/nb20_trend.json` (the *trend* — which way it's moving), and joins them into **one verdict record per hospital**.

**What it adds.** A per-hospital profit-margin characterization plus an explicit test of the ownership prediction (`fits_ownership_prediction`): does the margin picture match the naive for-profit-highest / public-thinnest story, or defy it the way the commercial-markup rung already did?

**Discipline carried forward.** Extends NB19 + NB20, never overwrites them (continuation pattern). Decision 44: a hospital-wide margin is never a per-procedure margin — enforced in the record contract, a live conflation sweep, and the integrity check. Decision 71: zero procedure codes anywhere in the emitted payload, labels included. Gated by `is_publishable()`; 0/5 real until the manual HCRIS pull (same download that unblocks NB19 and NB20) lands. Restart & Run All is canonical.

## Step 0 — Path resolver + `require()`
Resolve `ROOT` by checking which candidate directory actually contains `src/queries.py` (Day-21 fragility lesson — don't trust a single computed path). Define the `require()` run-order helper. No DB connection needed in this notebook.

In [ ]:
# Step 0 — path resolver + require()  (ROOT = the candidate that actually contains src/queries.py)

from pathlib import Path

_REPO_MARKER = Path("src") / "queries.py"   # the file that proves "this is the repo root"

def _resolve_root() -> Path:
    """Return the first candidate dir that actually contains src/queries.py."""
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]           # cwd, then walk up
    for cand in candidates:
        if (cand / _REPO_MARKER).is_file():
            return cand
    # Fail LOUDLY (Day-27 lesson): a clear error beats a garbage path.
    searched = "\n  ".join(str(c) for c in candidates)
    raise FileNotFoundError(
        "NB21 Step 0: could not locate src/queries.py from any parent of cwd.\n"
        f"  cwd = {here}\n"
        f"  searched:\n  {searched}\n"
        "  Fix: place this notebook inside the repo (…/dfw-hospital-pricing/notebooks/) "
        "and re-run — do not hardcode a path."
    )

ROOT = _resolve_root()
OUTPUTS = ROOT / "outputs"
NB19_JSON = OUTPUTS / "nb19_margin.json"
NB20_JSON = OUTPUTS / "nb20_trend.json"
NB21_JSON = OUTPUTS / "nb21_conclusion.json"   # NB21 writes here in Step F

def require(*names):
    """Run-order guard: assert each name is defined in globals (Restart & Run All is canonical)."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"NB21: {missing} not defined yet — an earlier cell hasn't run. "
            "Use Restart & Run All (single-cell runs trip run-order NameErrors by design)."
        )

# --- report what resolved (self-reporting, like NB20) ---
print("ROOT      :", ROOT)
print("OUTPUTS   :", OUTPUTS, "(exists)" if OUTPUTS.is_dir() else "(MISSING)")
print("NB19_JSON :", "found" if NB19_JSON.is_file() else "NOT FOUND", "->", NB19_JSON.name)
print("NB20_JSON :", "found" if NB20_JSON.is_file() else "NOT FOUND", "->", NB20_JSON.name)
print("NB21_JSON :", "will write ->", NB21_JSON.name)
print("Step 0 OK — paths resolved; require() ready. (No DB in NB21.)")

## Step 1 — Dual-payload recon
Load **both** `nb19_margin.json` and `nb20_trend.json`. Discover each schema rather than assuming it (recon-before-build). Confirm the same five CCNs appear in both parents, and report the gated state each closed at (NB19: 0/5 real; NB20: 0/15 real). This is where the join keys get pinned.

In [ ]:
# Step 1 — load nb19_margin.json + nb20_trend.json; discover schema; confirm 5 CCNs align across both

require("NB19_JSON", "NB20_JSON")   # Step 0 must have run
import json, math, re

CCN_RE = re.compile(r"^\d{6}$")   # a CCN is a 6-digit string

# --- Name<->CCN crosswalk (the project's known five). VERIFY these five by eye. ---
NAME_TO_CCN = {"Baylor":"450021","Methodist":"450051","Parkland":"450015",
               "MCA":"670103","THP":"450771"}
CCN_TO_NAME = {v: k for k, v in NAME_TO_CCN.items()}

def _load(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _is_real(rec):
    m = rec.get("margin", None)
    ok_m = isinstance(m, (int, float)) and not (isinstance(m, float) and math.isnan(m))
    return bool(rec.get("confirmed", False)) and ok_m

def _to_ccn(ident, rec=None):
    """Normalize any hospital identifier to a CCN: accept a real CCN, an in-record ccn, or a known name."""
    s = str(ident)
    if CCN_RE.match(s):                    return s
    if rec and CCN_RE.match(str(rec.get("ccn",""))): return str(rec["ccn"])
    if s in NAME_TO_CCN:                   return NAME_TO_CCN[s]
    raise KeyError(f"NB21 Step 1: identifier {s!r} is neither a CCN nor a known name "
                   f"(known: {sorted(NAME_TO_CCN)}). Update NAME_TO_CCN.")

# ---- NB19: per-hospital point records, normalized to CCN ----
raw19 = _load(NB19_JSON)
print("NB19 top-level keys:", list(raw19.keys()))
h19 = raw19.get("hospitals")
if not isinstance(h19, dict):
    raise KeyError("NB21 Step 1: expected NB19 'hospitals' dict — got " + repr(list(raw19.keys())))
sample_keys = sorted(next(iter(h19.values())).keys())
has_ccn_field19 = "ccn" in sample_keys
id_kind19 = ("in-record 'ccn' field" if has_ccn_field19
             else "hospital short-name key (no ccn field) -> mapped via NAME_TO_CCN")
nb19 = {}
for k, v in h19.items():
    c = _to_ccn(k, v)
    nb19[c] = dict(v); nb19[c]["_name"] = CCN_TO_NAME.get(c, str(k))
print(f"NB19: {len(nb19)} hospitals; identifier = {id_kind19}")
print("NB19 record keys (sample):", sample_keys)
nb19_real = sum(_is_real(r) for r in nb19.values())
print(f"NB19 gated state: {nb19_real}/{len(nb19)} real; publishable={raw19.get('publishable')}")

# ---- NB20: per-(ccn, year) trend records ----
raw20 = _load(NB20_JSON)
print("\nNB20 top-level keys:", list(raw20.keys()))
rec_list = rec_key = None
for k, v in raw20.items():
    if isinstance(v, list) and v and isinstance(v[0], dict):
        rec_list, rec_key = v, k; break
if rec_list is None:
    raise KeyError("NB21 Step 1: no records list in NB20 — keys " + repr(list(raw20.keys())))
if not all("ccn" in r for r in rec_list):
    raise KeyError("NB21 Step 1: NB20 records missing 'ccn' — cannot join")
for r in rec_list:                                   # validate every CCN is known
    if str(r["ccn"]) not in CCN_TO_NAME:
        raise KeyError(f"NB21 Step 1: NB20 CCN {r['ccn']!r} not in crosswalk — update NAME_TO_CCN.")
nb20_ccns  = sorted({str(r["ccn"]) for r in rec_list})
nb20_years = sorted({r.get("fiscal_year") for r in rec_list})
nb20_real  = sum(_is_real(r) for r in rec_list)
print(f"NB20: {len(rec_list)} records in '{rec_key}'; identifier = in-record 'ccn' field")
print(f"NB20: {len(nb20_ccns)} distinct CCNs x years {nb20_years}")
print(f"NB20 gated state: {nb20_real}/{len(rec_list)} real; publishable={raw20.get('publishable')}")

# ---- JOIN CHECK on normalized CCN ----
s19, s20 = set(nb19.keys()), set(nb20_ccns)
print("\nJoin check (normalized CCN):")
print("  NB19 ->", sorted((c, nb19[c]['_name']) for c in s19))
print("  NB20 ->", sorted(s20))
if s19 != s20:
    print("  ⚠ MISMATCH — only-in-NB19:", sorted(s19 - s20), "| only-in-NB20:", sorted(s20 - s19))
    raise ValueError("NB21 Step 1: CCN sets differ across parents even after crosswalk — resolve before Step 2.")
JOIN_KEYS   = sorted(s19)
NB19_BY_CCN = nb19
NB20_RECORDS = rec_list
print(f"  ✅ aligned on {len(JOIN_KEYS)} CCNs -> JOIN_KEYS:", JOIN_KEYS)
print("     names:", [CCN_TO_NAME[c] for c in JOIN_KEYS])
print("\nStep 1 OK — parents joined via NAME_TO_CCN; CCN is the canonical key, short name carried for display.")

## Step 2 — Synthesis contract
`make_verdict_record`, `validate_verdict_record` on a **per-hospital** grain (one row per CCN). Joins the NB19 point margin with the NB20 trend direction, attaches ownership + county regime, and derives `fits_ownership_prediction`. Decision-44 tripwire rejects any record carrying `cpt`/`hcpcs`/`code`/`procedure`/`73721` as key or value.

In [ ]:
# Step 2 — make_verdict_record / validate_verdict_record (per-hospital grain; D44 tripwire; ownership-prediction test)

require("JOIN_KEYS", "NB19_BY_CCN", "NB20_RECORDS", "CCN_TO_NAME")
import math

# --- County / funding-regime map (project structural layer; per CCN). VERIFY by eye. ---
COUNTY_REGIME = {
    "450021": {"county":"Dallas",  "safety_net":"Parkland (tax-funded public)", "regime":"safety-net present"},
    "450051": {"county":"Dallas",  "safety_net":"Parkland (tax-funded public)", "regime":"safety-net present"},
    "450015": {"county":"Dallas",  "safety_net":"is the safety net",            "regime":"is safety-net"},
    "670103": {"county":"Tarrant", "safety_net":"JPS (tax-funded public)",      "regime":"safety-net present"},
    "450771": {"county":"Collin",  "safety_net":"none",                          "regime":"no public hospital"},
}

# --- The naive ownership prediction being TESTED (not asserted): --------------------
#   for-profit -> highest margin ; public/safety-net -> thinnest ; nonprofit -> interior.
#   fits = does the hospital sit where its ownership predicts, RELATIVE to the others.
OWNERSHIP_PREDICTION = "for-profit highest, public thinnest, nonprofit interior"
TREND_FLAT_EPS = 0.5   # percentage points; |latest-earliest| below this reads as 'flat'

# Decision-44 tripwire: no procedure identifier may live in a margin record.
_BANNED = ("cpt", "hcpcs", "procedure", "73721")   # 'code' handled separately (substring-safe)
def _d44_scan(rec):
    for k, v in rec.items():
        kl = str(k).lower()
        if kl == "code" or any(t in kl for t in _BANNED):
            return f"banned key {k!r}"
        if isinstance(v, str):
            vl = v.lower()
            if "73721" in vl or "hcpcs" in vl or "cpt " in vl or vl.strip() == "cpt":
                return f"banned token in value of {k!r}: {v!r}"
    return None

def _f(x):
    """NaN-safe float: None/NaN -> nan; numbers pass through."""
    if x is None: return math.nan
    if isinstance(x, float) and math.isnan(x): return math.nan
    if isinstance(x, (int, float)): return float(x)
    return math.nan

def reduce_trend(series):
    """series = {fiscal_year: margin}. Reduce to (direction, change) over REAL points only.
       <2 real points -> (None, nan): never fabricate a direction."""
    pts = sorted((y, _f(m)) for y, m in series.items())
    real = [(y, m) for y, m in pts if not math.isnan(m)]
    if len(real) < 2:
        return None, math.nan
    change = real[-1][1] - real[0][1]
    if abs(change) < TREND_FLAT_EPS: direction = "flat"
    elif change > 0:                 direction = "rising"
    else:                            direction = "declining"
    return direction, round(change, 2)

def make_verdict_record(ccn, name, nb19_rec, nb20_series):
    """One per-hospital verdict: join NB19 point + NB20 trend + context. fits filled later (cross-sectional)."""
    reg = COUNTY_REGIME.get(ccn, {"county":None,"safety_net":None,"regime":None})
    own19 = nb19_rec.get("ownership")
    point = _f(nb19_rec.get("margin"))
    direction, change = reduce_trend(nb20_series)
    # confirmed only if BOTH lineages confirmed AND a real point exists (two-flag sourcing)
    confirmed = bool(nb19_rec.get("confirmed", False)) and not math.isnan(point)
    corroborated = bool(nb19_rec.get("corroborated", False))
    rec = {
        "ccn": ccn, "name": name, "ownership": own19,
        "county": reg["county"], "safety_net": reg["safety_net"], "funding_regime": reg["regime"],
        "point_margin": point, "point_fiscal_year": nb19_rec.get("fiscal_year"),
        "trend_direction": direction, "trend_change_pp": change,
        "basis": nb19_rec.get("basis", "operating"),
        "verdict": None,                      # plain-language, filled only when publishable
        "fits_ownership_prediction": None,    # cross-sectional, filled by assess_ownership_fit
        "confirmed": confirmed, "corroborated": corroborated,
        "level": "hospital",                  # Decision 44
        "parents": ["NB19 nb19_margin.json", "NB20 nb20_trend.json"],
        "source": nb19_rec.get("source"),
    }
    return rec

def validate_verdict_record(rec):
    """Grain + D44 + required-field checks. Returns list of failures (empty = ok)."""
    fails = []
    if rec.get("level") != "hospital":
        fails.append(f"{rec.get('ccn')}: level != hospital")
    for req in ("ccn","name","ownership","point_margin","trend_direction","basis","parents"):
        if req not in rec: fails.append(f"{rec.get('ccn')}: missing {req}")
    hit = _d44_scan(rec)
    if hit: fails.append(f"{rec.get('ccn')}: D44 tripwire — {hit}")
    if not (isinstance(rec.get("point_margin"), float)):
        fails.append(f"{rec.get('ccn')}: point_margin must be float (NaN ok)")
    return fails

def assess_ownership_fit(verdicts):
    """Cross-sectional test. Fills fits_ownership_prediction only when ALL points are real; else None."""
    vals = {c: v["point_margin"] for c, v in verdicts.items()}
    if any(math.isnan(x) for x in vals.values()):
        for v in verdicts.values(): v["fits_ownership_prediction"] = None
        return False   # not assessable yet
    for c, v in verdicts.items():
        x = vals[c]; own = v["ownership"]
        lower  = any(y < x for k, y in vals.items() if k != c)
        higher = any(y > x for k, y in vals.items() if k != c)
        if   own == "for-profit": v["fits_ownership_prediction"] = (not higher)   # (tied) top
        elif own == "public":     v["fits_ownership_prediction"] = (not lower)    # (tied) bottom
        else:                     v["fits_ownership_prediction"] = (lower and higher)  # interior
    return True

# ---------------- inline self-checks ----------------
print("reduce_trend rising :", reduce_trend({2022:1.0, 2023:2.0, 2024:5.0}))
print("reduce_trend NaN-gap:", reduce_trend({2022:1.0, 2023:None, 2024:None}))
_demo = make_verdict_record("670103","MCA",
        {"ownership":"for-profit","fiscal_year":2023,"margin":None,"basis":"operating",
         "confirmed":False,"corroborated":True,"source":"HCRIS (PENDING)"},
        {2022:None,2023:None,2024:None})
print("demo verdict (gated):", {k:_demo[k] for k in
      ("ccn","ownership","point_margin","trend_direction","confirmed","fits_ownership_prediction","level")})
print("validate clean rec  :", validate_verdict_record(_demo) or "OK")
_bad = dict(_demo); _bad["verdict"] = "margin on 73721 knee MRI"
print("validate dirty rec  :", validate_verdict_record(_bad))
_synx = {
  "670103":{"ownership":"for-profit","point_margin":12.0},
  "450021":{"ownership":"nonprofit","point_margin":8.0},
  "450051":{"ownership":"nonprofit","point_margin":6.0},
  "450771":{"ownership":"nonprofit","point_margin":5.0},
  "450015":{"ownership":"public","point_margin":-1.0},
}
ok = assess_ownership_fit(_synx)
print("assess (naive holds):", ok, {c:_synx[c]["fits_ownership_prediction"] for c in _synx})
_synx2 = {
  "450021":{"ownership":"nonprofit","point_margin":12.0},
  "670103":{"ownership":"for-profit","point_margin":7.0},
  "450051":{"ownership":"nonprofit","point_margin":6.0},
  "450771":{"ownership":"nonprofit","point_margin":5.0},
  "450015":{"ownership":"public","point_margin":-1.0},
}
assess_ownership_fit(_synx2)
print("assess (inverted)   :", {c:_synx2[c]["fits_ownership_prediction"] for c in _synx2})
print("\nStep 2 OK — contract defined; tripwire fires; ownership-fit test works both directions.")

## Step 3 — Synthetic smoke test  `[SYNTHETIC DATA]`
Isolated `SYN_` roster whose point+trend pairs map to the four real situations (rising nonprofit, high-but-declining for-profit, thin-negative public, and a NaN-gap hospital). Joins synthetic point to synthetic trend into synthetic verdicts. All rows watermarked and unconfirmed; `REAL_VERDICT` isolation asserted.

In [ ]:
# Step 3 — synthetic smoke (SYN_ roster, [SYNTHETIC DATA]); four point+trend pairs -> four verdicts

require("make_verdict_record", "assess_ownership_fit", "validate_verdict_record", "JOIN_KEYS")
WATERMARK = "[SYNTHETIC DATA]"

# Isolated SYN_ roster (non-real ids) mirroring the real ownership mix, exercising all
# four reduce_trend outcomes: rising / flat / declining / None(NaN-gap).
SYN_NB19 = {   # synthetic point rows (shape of an NB19 record)
  "SYN_FP1": {"ownership":"for-profit","fiscal_year":2024,"margin":11.0,"basis":"operating","confirmed":False,"corroborated":True,"source":"SYN"},
  "SYN_PUB1":{"ownership":"public",    "fiscal_year":2024,"margin":-1.5,"basis":"operating","confirmed":False,"corroborated":True,"source":"SYN"},
  "SYN_NP1": {"ownership":"nonprofit", "fiscal_year":2024,"margin":6.0, "basis":"operating","confirmed":False,"corroborated":True,"source":"SYN"},
  "SYN_NP2": {"ownership":"nonprofit", "fiscal_year":2024,"margin":4.0, "basis":"operating","confirmed":False,"corroborated":True,"source":"SYN"},
  "SYN_NP3": {"ownership":"nonprofit", "fiscal_year":2024,"margin":5.0, "basis":"operating","confirmed":False,"corroborated":True,"source":"SYN"},
}
SYN_SERIES = {  # synthetic 3-year trend series
  "SYN_FP1":  {2022:14.0, 2023:12.5, 2024:11.0},   # declining (for-profit, high-but-falling)
  "SYN_PUB1": {2022:-0.5, 2023:-1.0, 2024:-1.5},   # declining (public, thin-negative)
  "SYN_NP1":  {2022:3.0,  2023:4.5,  2024:6.0},    # rising
  "SYN_NP2":  {2022:4.2,  2023:3.9,  2024:4.0},    # flat (|change| < eps)
  "SYN_NP3":  {2022:5.0,  2023:None, 2024:None},   # NaN-gap -> <2 real pts -> direction None
}

SYN_VERDICTS = {}
for cid, row in SYN_NB19.items():
    rec = make_verdict_record(cid, cid.replace("SYN_",""), row, SYN_SERIES[cid])
    rec["synthetic"] = True
    rec["watermark"] = WATERMARK
    SYN_VERDICTS[cid] = rec

# cross-sectional ownership-fit on the synthetic set (all points real -> assessable)
syn_assessed = assess_ownership_fit(SYN_VERDICTS)

# validate every synthetic record against the contract
syn_fails = []
for rec in SYN_VERDICTS.values():
    syn_fails += validate_verdict_record(rec)

# ---- isolation guards (NB20 discipline: synthetic must never touch real) ----
real_ccns = set(JOIN_KEYS)
assert all(cid.startswith("SYN_") for cid in SYN_VERDICTS), "synthetic ids must be SYN_-prefixed"
assert not (set(SYN_VERDICTS) & real_ccns), "synthetic roster leaked a real CCN!"
assert all(r["watermark"] == WATERMARK and r["synthetic"] for r in SYN_VERDICTS.values()), "watermark missing"
assert all(r["confirmed"] is False for r in SYN_VERDICTS.values()), "synthetic must stay unconfirmed"
REAL_VERDICTS = {}   # built in Step R — kept empty & separate here

# ---- report ----
print(f"{'id':9} {'own':11} {'point':>6} {'trend':>10} {'chg':>6} {'fits':>6}  wm")
for cid, r in SYN_VERDICTS.items():
    print(f"{cid:9} {r['ownership']:11} {r['point_margin']:6.1f} "
          f"{str(r['trend_direction']):>10} {str(r['trend_change_pp']):>6} "
          f"{str(r['fits_ownership_prediction']):>6}  {r['watermark']}")
print(f"\nassess_ownership_fit assessable? {syn_assessed} (expect True — all synthetic points real)")
print("validate synthetic set:", syn_fails or "OK (all pass contract)")
print("isolation: SYN ids non-real ✓ | watermarked ✓ | unconfirmed ✓ | REAL_VERDICTS empty ✓")
print("\nStep 3 OK — full join->reduce->assess->validate chain works on watermarked synthetic; real untouched.")

## Step R — Real assembly + gate
Build the real verdicts by joining the real NB19 records to the real NB20 records on CCN. No new dollars are pulled here — NB21 inherits whatever NB19/NB20 hold, so today every field is a NaN sentinel and the join emits 0/5 real. `is_publishable()` three-lock (synthetic taint / `MARGINS_PUBLISHABLE` / per-record `confirmed`). Prints the unblock recipe.

In [ ]:
# Step R — join real NB19 x NB20 on CCN; is_publishable() three-lock; 0/5 real today; print unblock recipe

require("JOIN_KEYS","NB19_BY_CCN","NB20_RECORDS","CCN_TO_NAME","make_verdict_record",
        "assess_ownership_fit","validate_verdict_record","raw19","raw20")
import math

# group NB20 records -> per-CCN series {year: margin} + per-CCN confirmation & coverage
nb20_series_by_ccn    = {c: {} for c in JOIN_KEYS}
nb20_confirmed_by_ccn = {c: True for c in JOIN_KEYS}
for r in NB20_RECORDS:
    c = str(r["ccn"])
    if c not in nb20_series_by_ccn:      # already validated in Step 1, belt-and-suspenders
        continue
    nb20_series_by_ccn[c][r.get("fiscal_year")] = r.get("margin")
    nb20_confirmed_by_ccn[c] = nb20_confirmed_by_ccn[c] and bool(r.get("confirmed", False))

# build the real per-hospital verdicts through the SAME contract as Step 3
REAL_VERDICTS = {}
for c in JOIN_KEYS:
    nb19_rec = NB19_BY_CCN[c]
    name = nb19_rec.get("_name", CCN_TO_NAME.get(c, c))
    rec = make_verdict_record(c, name, nb19_rec, nb20_series_by_ccn[c])
    # a verdict is confirmed only if BOTH lineages are confirmed (point AND every trend year)
    rec["confirmed"] = bool(rec["confirmed"]) and nb20_confirmed_by_ccn[c]
    rec["trend_years_present"] = sorted(y for y in nb20_series_by_ccn[c] if y is not None)
    REAL_VERDICTS[c] = rec

# cross-sectional ownership-fit (None today: all points NaN -> not assessable)
assessable = assess_ownership_fit(REAL_VERDICTS)

# validate + no-synthetic-leak
real_fails = []
for rec in REAL_VERDICTS.values():
    real_fails += validate_verdict_record(rec)
assert not any(v.get("synthetic") for v in REAL_VERDICTS.values()), "synthetic leaked into REAL_VERDICTS!"

# ---------------- gate: three locks ----------------
PARENTS_PUBLISHABLE = bool(raw19.get("publishable")) and bool(raw20.get("publishable"))
MARGINS_PUBLISHABLE = False   # NB21 master switch — flip ONLY after real data lands + eyeball

def is_publishable(verdicts):
    lock_no_synth  = not any(v.get("synthetic") for v in verdicts.values())
    lock_switch    = bool(MARGINS_PUBLISHABLE)
    lock_confirmed = len(verdicts) > 0 and all(v.get("confirmed") for v in verdicts.values())
    ok = lock_no_synth and lock_switch and lock_confirmed
    return ok, {"no_synthetic_taint":lock_no_synth,
                "MARGINS_PUBLISHABLE":lock_switch,
                "all_confirmed":lock_confirmed}

PUBLISHABLE, LOCKS = is_publishable(REAL_VERDICTS)
n_real = sum(1 for v in REAL_VERDICTS.values()
             if v.get("confirmed") and isinstance(v["point_margin"], float)
             and not math.isnan(v["point_margin"]))

# ---------------- report ----------------
print(f"{'ccn':7} {'name':10} {'own':11} {'point':>6} {'trend':>10} {'fits':>5} {'conf':>5}")
for c in JOIN_KEYS:
    v = REAL_VERDICTS[c]
    pm = "nan" if math.isnan(v["point_margin"]) else f"{v['point_margin']:.1f}"
    print(f"{c:7} {v['name']:10} {v['ownership']:11} {pm:>6} "
          f"{str(v['trend_direction']):>10} {str(v['fits_ownership_prediction']):>5} {str(v['confirmed']):>5}")
print("\nvalidate REAL set:", real_fails or "OK (all pass contract)")
print("ownership-fit assessable?", assessable, "(expect False today — points are NaN)")
print(f"parents publishable? NB19+NB20 = {PARENTS_PUBLISHABLE}")
print("is_publishable() =", PUBLISHABLE, "| locks:", LOCKS)
print(f"REAL: {n_real}/{len(REAL_VERDICTS)} real")

print("\n--- UNBLOCK RECIPE (NB21 derives everything; it has NO data slot of its own) ---")
print("  1. Do the manual HCRIS pull (same one NB19 & NB20 need).")
print("  2. Fill NB19 REAL_INPUTS (single-year) + NB20 REAL_INPUTS (per-year FY2022-24); Restart&RunAll both.")
print("     -> their JSONs regenerate with real margins + confirmed=True.")
print("  3. Set MARGINS_PUBLISHABLE=True here, then Restart&RunAll NB21.")
print("     -> point, trend, and fits_ownership_prediction all fill; is_publishable() flips itself.")
print("\nStep R OK — real verdicts assembled via the Step-2 contract; gated at 0/5, is_publishable() False.")

## Step S — Sensitivity
Verdict robustness toggles: (a) does the trend-direction call flip under latest-year vs multi-year mean? (b) does `fits_ownership_prediction` hold under operating vs total basis? (c) live Decision-44 conflation sweep across all verdict records. Headline `REAL_VERDICT` untouched; mechanism previewed on synthetic where real is PENDING.

In [ ]:
# Step S — direction-call & ownership-prediction robustness toggles + live D44 sweep

require("SYN_VERDICTS","SYN_SERIES","REAL_VERDICTS","assess_ownership_fit","_d44_scan","reduce_trend")
import math, statistics as _stats

# snapshot the real headline so we can prove Step S never mutates it
_headline_before = {c:(REAL_VERDICTS[c]["trend_direction"],
                       REAL_VERDICTS[c]["fits_ownership_prediction"]) for c in REAL_VERDICTS}

# --- Toggle 1: trend REDUCTION method (latest-year vs multi-year mean vs direction) ---
#     Different reductions can disagree; we preview on synthetic since real is NaN.
print("Toggle 1 — reduction method (synthetic preview):")
print(f"  {'id':9} {'latest':>7} {'mean':>7} {'direction':>10}")
for cid, series in SYN_SERIES.items():
    real_pts = [m for _, m in sorted(series.items()) if isinstance(m,(int,float))]
    latest = real_pts[-1] if real_pts else math.nan
    mean   = round(_stats.mean(real_pts),2) if real_pts else math.nan
    d, _   = reduce_trend(series)
    print(f"  {cid:9} {latest:>7} {mean:>7} {str(d):>10}")
print("  -> latest and mean disagree whenever the series moves (declining: latest<mean).")

# --- Toggle 2: operating vs TOTAL basis (total flatters the tax-funded public case) ---
#     Mirror of NB20 Decision 70: total adds non-operating income -> public looks better.
_op = {"FP":12.0, "NP_a":8.0, "NP_b":6.0, "NP_c":5.0, "PUB":-1.0}      # operating
_tot= {"FP":12.5, "NP_a":8.3, "NP_b":6.2, "NP_c":5.1, "PUB":15.0}      # +non-operating income
def _fit_pattern(vals, owns):
    d = {k:{"point_margin":vals[k], "ownership":owns[k]} for k in vals}
    assess_ownership_fit(d)
    return {k:d[k]["fits_ownership_prediction"] for k in d}
_owns = {"FP":"for-profit","NP_a":"nonprofit","NP_b":"nonprofit","NP_c":"nonprofit","PUB":"public"}
op_fit  = _fit_pattern(_op,  _owns)
tot_fit = _fit_pattern(_tot, _owns)
print("\nToggle 2 — operating vs total basis (synthetic preview):")
print(f"  public margin: operating {_op['PUB']:+.1f}  vs  total {_tot['PUB']:+.1f}  "
      f"(total flatters by {_tot['PUB']-_op['PUB']:+.1f}pp)")
print(f"  public fits-as-thinnest:  operating={op_fit['PUB']}  vs  total={tot_fit['PUB']}")
print("  -> total masks the safety-net hospital's structural position; operating stays the headline.")

# --- Toggle 3: LIVE Decision-44 conflation sweep across ALL verdict records ---
_all = list(REAL_VERDICTS.values()) + list(SYN_VERDICTS.values())
_viol = [(r.get("ccn"), _d44_scan(r)) for r in _all if _d44_scan(r)]
print(f"\nToggle 3 — live D44 sweep: {len(_all)} records "
      f"({len(REAL_VERDICTS)} real + {len(SYN_VERDICTS)} synthetic) -> {len(_viol)} violations")

# --- headline untouched? ---
_headline_after = {c:(REAL_VERDICTS[c]["trend_direction"],
                      REAL_VERDICTS[c]["fits_ownership_prediction"]) for c in REAL_VERDICTS}
assert _headline_before == _headline_after, "Step S mutated the real headline!"
print("\nreal headline REAL_VERDICTS untouched by Step S ✓")
print("Step S OK — reduction & basis mechanisms previewed on synthetic; D44 sweep clean; real PENDING.")

## Step V — Chart
The bookend visual: each hospital placed by **point margin** against **trend direction**, colored by ownership, PENDING hospitals drawn as hatched ghosts (not zero-value points). Watermark driven off `is_publishable()` ("DRAFT — NOT PUBLISHABLE" today). Title states hospital-wide operating-margin verdict, **NOT per-procedure**, so a screenshot can't be misread.

In [ ]:
# Step V — point-vs-trend placement by ownership; PENDING ghosts; watermark off is_publishable(); D44-safe title

require("REAL_VERDICTS","PUBLISHABLE","CCN_TO_NAME","JOIN_KEYS")
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OWN_COLOR = {"for-profit":"#c1440e", "nonprofit":"#2a6f97", "public":"#4c956c"}
_DIR_Y = {"declining":-1, "flat":0, "rising":1}

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.axhline(0, color="#cccccc", lw=0.8, zorder=1)
ax.axvline(0, color="#cccccc", lw=0.8, ls=":", zorder=1)

placed, pending = [], []
for c in JOIN_KEYS:
    v = REAL_VERDICTS[c]
    x = v["point_margin"]; d = v["trend_direction"]
    is_pending = (not isinstance(x,(int,float))) or math.isnan(x) or (d is None)
    (pending if is_pending else placed).append((c, v))

# PENDING -> hatched hollow grey ghosts, stacked on the x=0 baseline (not zero-VALUE points)
for i, (c, v) in enumerate(pending):
    yy = (i - (len(pending)-1)/2) * 0.45          # spread so labels don't collide
    ax.scatter(0, yy, s=340, facecolors="none", edgecolors="#9aa0a6",
               hatch="////", linewidths=1.4, zorder=3)
    ax.annotate(f"{v['name']} · {v['ownership']}", (0, yy), xytext=(12, 0),
                textcoords="offset points", va="center", fontsize=9, color="#5f6368")

# PLACED (real) -> filled marker at (point, direction), colored by ownership
for c, v in placed:
    x = v["point_margin"]; y = _DIR_Y.get(v["trend_direction"], 0)
    ax.scatter(x, y, s=320, color=OWN_COLOR.get(v["ownership"],"#666"),
               edgecolors="black", linewidths=0.6, zorder=4)
    fit = v.get("fits_ownership_prediction")
    tag = "" if fit is None else ("  ✓fits" if fit else "  ✗defies")
    ax.annotate(f"{v['name']}{tag}", (x, y), xytext=(8, 8),
                textcoords="offset points", fontsize=9)

ax.set_yticks([-1,0,1]); ax.set_yticklabels(["declining","flat","rising"])
ax.set_xlabel("hospital-wide operating margin  (%, point — from NB19)")
ax.set_ylabel("multi-year trend  (from NB20)")
ax.set_title("Profit-margin verdict per hospital: point × trend  "
             "(hospital-wide operating margin — NOT per-procedure)", fontsize=11)
if pending:
    ax.text(0.02, 0.02, f"{len(pending)} pending — become placed markers when real data lands",
            transform=ax.transAxes, fontsize=8, color="#9aa0a6", style="italic")
ax.set_xlim(-6, 16); ax.set_ylim(-1.6, 1.6)   # placeholder range; auto-widen when real lands

legend = [Patch(facecolor=col, edgecolor="black", label=own) for own,col in OWN_COLOR.items()]
legend.append(Patch(facecolor="none", edgecolor="#9aa0a6", hatch="////", label="pending (no data)"))
ax.legend(handles=legend, loc="upper left", fontsize=8, framealpha=0.9)

# watermark driven off is_publishable() — DRAFT today
if not PUBLISHABLE:
    ax.text(0.5, 0.5, "DRAFT — NOT PUBLISHABLE", transform=ax.transAxes,
            fontsize=30, color="red", alpha=0.16, ha="center", va="center", rotation=24, zorder=5)

plt.tight_layout()
plt.show()
print(f"Step V OK — chart rendered. placed={len(placed)} pending={len(pending)}; "
      f"watermark={'DRAFT' if not PUBLISHABLE else 'none'} (is_publishable={PUBLISHABLE}).")

## Step F — `NB21_CONCLUSION` handoff
One JSON-friendly payload (NaN->None), every record tagged `level:"hospital"` and `parents:["NB19 ...","NB20 ..."]` (extends, never replaces). App label + do-not-publish block + unblock recipe. Decision-71 literal guard (no procedure code anywhere in the blob, label included). Persist to `outputs/nb21_conclusion.json`; verify round-trip.

In [ ]:
# Step F — NB21_CONCLUSION payload (NaN->None, level=hospital, parents=[NB19,NB20]); D71 guard; persist + round-trip

require("REAL_VERDICTS","JOIN_KEYS","PUBLISHABLE","LOCKS","NB21_JSON",
        "OWNERSHIP_PREDICTION","validate_verdict_record","raw19","raw20")
import json, math

def _nan_to_none(o):
    if isinstance(o, float) and math.isnan(o): return None
    if isinstance(o, dict):  return {k:_nan_to_none(v) for k,v in o.items()}
    if isinstance(o, list):  return [_nan_to_none(v) for v in o]
    return o

# assemble per-hospital records (JSON-safe, level=hospital, no internal keys leaking)
records = []
for c in JOIN_KEYS:
    v = dict(REAL_VERDICTS[c])
    v.pop("_name", None)                       # internal only; 'name' already carries it
    v["level"] = "hospital"                     # Decision 44 — belt & suspenders
    v["parents"] = ["nb19_margin.json", "nb20_trend.json"]
    records.append(_nan_to_none(v))

n_real = sum(1 for r in records if r["confirmed"] and r["point_margin"] is not None)

# app label deliberately references NO procedure (hospital-wide margin needs none) -> Decision 71 clean
LABEL = ("Per-hospital profit-margin verdict: hospital-wide operating margin (point, from cost "
         "reports) and its multi-year direction. Hospital-level only — never a per-procedure margin.")

NB21_CONCLUSION = {
    "notebook": "NB21",
    "factor": "profit_margin",
    "lens": "conclusion — point x trend bookend (extends NB19 + NB20, replaces neither)",
    "parents": ["nb19_margin.json", "nb20_trend.json"],
    "level": "hospital",
    "basis_headline": "operating",
    "ownership_prediction_tested": OWNERSHIP_PREDICTION,
    "label": LABEL,
    "vintage": {"nb19": raw19.get("vintage"), "nb20": raw20.get("vintage")},
    "publishable": bool(PUBLISHABLE),
    "counts": {"hospitals": len(records), "real": n_real, "pending": len(records)-n_real},
    "records": records,
    "do_not_publish": {
        "reason": "0 real margins; inherits NB19+NB20 gated state; MARGINS_PUBLISHABLE=False.",
        "locks": LOCKS,
        "unblock": [
            "Do the manual HCRIS pull (same one NB19 & NB20 need).",
            "Fill NB19 REAL_INPUTS (single-year) + NB20 REAL_INPUTS (per-year FY2022-24); Restart&RunAll both.",
            "Set MARGINS_PUBLISHABLE=True in NB21 Step R; Restart&RunAll NB21.",
        ],
    },
}

# ---- Decision-71 literal guard: NO procedure code anywhere in the serialized payload ----
blob = json.dumps(NB21_CONCLUSION)
for tok in ("73721", "hcpcs", " cpt", "cpt "):
    assert tok not in blob.lower(), f"Decision-71 guard: banned token {tok!r} found in NB21 payload"
# every record must still pass the contract — validate the in-memory (float-NaN) form,
# NOT the serialized (None) form: JSON turns NaN->None, which is not a float.
_revalidate = []
for c in JOIN_KEYS:
    _revalidate += validate_verdict_record(REAL_VERDICTS[c])
assert not _revalidate, f"Step F: record validation failed: {_revalidate}"

# ---- persist + round-trip ----
with open(NB21_JSON, "w", encoding="utf-8") as f:
    json.dump(NB21_CONCLUSION, f, indent=2)
with open(NB21_JSON, "r", encoding="utf-8") as f:
    _rt = json.load(f)
assert _rt == NB21_CONCLUSION, "round-trip mismatch!"

print(f"NB21_CONCLUSION written -> {NB21_JSON.name}")
print(f"  hospitals={NB21_CONCLUSION['counts']['hospitals']} "
      f"real={NB21_CONCLUSION['counts']['real']} pending={NB21_CONCLUSION['counts']['pending']}")
print(f"  publishable={NB21_CONCLUSION['publishable']}  D71 guard: clean  round-trip: OK")
print("  sample record:", json.dumps(records[0], indent=None)[:180], "...")
print("\nStep F OK — bookend payload persisted; NaN->None; level=hospital; parents=[NB19,NB20]; D71 clean.")

## Step I — Integrity sweep
Collect-all-failures guards: roster (5 hospitals), per-hospital grain, no-synthetic-leak, SYN-watermark, three Decision-44 checks (record / level-tag / serialized-payload), join-consistency (every hospital present in **both** parents), two publish-state checks, margin-sanity. Close with the honest one-line status.

In [ ]:
# Step I — integrity sweep (roster, grain, no-leak, SYN-watermark, 3x D44, join-consistency, publish-state, sanity)

require("REAL_VERDICTS","SYN_VERDICTS","JOIN_KEYS","NB19_BY_CCN","nb20_series_by_ccn",
        "PUBLISHABLE","LOCKS","MARGINS_PUBLISHABLE","_d44_scan","NB21_JSON")
import json, math

fails = []
def check(cond, msg):
    if not cond: fails.append(msg)

# reload the persisted payload for the serialized-form checks
with open(NB21_JSON, "r", encoding="utf-8") as f:
    saved = json.load(f)
saved_blob = json.dumps(saved).lower()

# 1. roster — exactly the five known CCNs
check(sorted(REAL_VERDICTS) == sorted(JOIN_KEYS) and len(JOIN_KEYS) == 5,
      f"roster: expected 5 known CCNs, got {sorted(REAL_VERDICTS)}")
# 2. grain — one record per CCN, level=hospital
check(all(v["level"] == "hospital" for v in REAL_VERDICTS.values()),
      "grain: a real record is not level=hospital")
check(len(saved["records"]) == 5, f"grain: payload has {len(saved['records'])} records, expected 5")
# 3. no-synthetic-leak — nothing synthetic in the real set or the payload
check(not any(v.get("synthetic") for v in REAL_VERDICTS.values()), "leak: synthetic in REAL_VERDICTS")
check("[synthetic data]" not in saved_blob, "leak: [SYNTHETIC DATA] watermark in persisted payload")
# 4. SYN-watermark — every synthetic row properly stamped & unconfirmed
check(all(v.get("watermark") == "[SYNTHETIC DATA]" and v.get("synthetic") and v["confirmed"] is False
          for v in SYN_VERDICTS.values()), "SYN-watermark: a synthetic row is mis-stamped")
# 5. D44 (record-level) — no procedure token in any real record
check(not any(_d44_scan(v) for v in REAL_VERDICTS.values()), "D44: procedure token in a real record")
# 6. D44 (level-tag) — every persisted record tagged hospital
check(all(r["level"] == "hospital" for r in saved["records"]), "D44: a persisted record not level=hospital")
# 7. D44 (serialized payload) — literal scan of the saved blob
check(all(tok not in saved_blob for tok in ("73721","hcpcs"," cpt","cpt ")),
      "D44: procedure code in serialized payload")
# 8. basis-labeled — every record carries a basis; headline declared
check(all("basis" in v for v in REAL_VERDICTS.values()) and saved.get("basis_headline") == "operating",
      "basis: missing basis on a record or headline not 'operating'")
# 9. join-consistency — every hospital present in BOTH parents
check(all(c in NB19_BY_CCN for c in JOIN_KEYS), "join: a CCN missing from NB19")
check(all(c in nb20_series_by_ccn and len(nb20_series_by_ccn[c]) > 0 for c in JOIN_KEYS),
      "join: a CCN missing an NB20 series")
# 10. publish-state — gated correctly today
check(PUBLISHABLE is False and MARGINS_PUBLISHABLE is False and LOCKS["all_confirmed"] is False,
      "publish-state: expected fully gated (is_publishable False)")
check(saved["publishable"] is False and saved["counts"]["real"] == 0,
      "publish-state: persisted payload not gated at 0 real")
# 11. margin-sanity — any REAL point_margin must be in a sane band; NaN/None allowed
for c, v in REAL_VERDICTS.items():
    x = v["point_margin"]
    if isinstance(x,(int,float)) and not (isinstance(x,float) and math.isnan(x)):
        check(-60.0 <= x <= 60.0, f"sanity: {c} point_margin {x} out of [-60,60]")

# ---- verdict ----
n_real = saved["counts"]["real"]; n_pending = saved["counts"]["pending"]
if fails:
    print("Integrity: FAIL —", len(fails), "issue(s):")
    for m in fails: print("  -", m)
else:
    print(f"Integrity: OK — 5 hospitals · 5 verdicts · {n_real} real · {n_pending} pending · "
          f"MARGINS_PUBLISHABLE={MARGINS_PUBLISHABLE}")
    print("  (correctly gated, NOT ship-ready — lights up when the HCRIS pull fills NB19 + NB20.)")